# CA2 – Modelling and Evaluation
**Moustafa Almohamed – X00228076**

AI: Theory to Practice (COMPH3004) – UCI Bank Marketing dataset

## 1. Setup and Data Loading
This notebook continues from CA1. It loads the data, repeats the CA1 preprocessing, and then builds and evaluates the models.

In [4]:
# tools for tables and numbers
import pandas as pd
import numpy as np

# load the bank data (columns are separated by ";")
df = pd.read_csv("data/bank-additional-full.csv", sep=";")
print(df.shape)  # (rows, columns)

(41188, 21)


## 2. Cleaning (same steps as CA1)
The cleaning decisions from CA1 are repeated here so the notebook runs on its own.

In [5]:
# remove the 12 exact duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

# pdays = 999 means "never contacted", so turn it into a yes/no column
df["prev_contacted"] = (df["pdays"] != 999).astype(int)

# drop pdays (replaced), duration (leakage) and default (almost all "no")
df = df.drop(columns=["pdays", "duration", "default"])

# "unknown" becomes empty (NaN) so the pipeline can fill it later
# housing and loan keep "unknown" as its own category
unk_cols = ["job", "marital", "education"]
df[unk_cols] = df[unk_cols].replace("unknown", np.nan)

print(df.shape)

(41176, 19)


## 3. Train/Test Split
The data is split 80/20 with the same settings as CA1. The test set is kept aside until the final evaluation.

In [6]:
from sklearn.model_selection import train_test_split

# X = the inputs, y = the answer as 1 (yes) or 0 (no)
X = df.drop(columns="y")
y = (df["y"] == "yes").astype(int)

# 80% train, 20% test, same yes/no mix in both (same split as CA1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Share of yes in train:", round(y_train.mean(), 3))

Train: (32940, 18)  Test: (8236, 18)
Share of yes in train: 0.113


## 4. Preprocessing Pipeline (from CA1)
The CA1 pipeline is fitted once on the training set and then applied to the test set. It gives 21 ready columns for all models.

In [7]:
from functools import partial
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectPercentile, SequentialFeatureSelector, mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler

# the column groups
num_cols = ["age", "campaign", "previous", "emp.var.rate", "cons.price.idx",
            "cons.conf.idx", "euribor3m", "nr.employed"]
ordinal_cols = ["education"]
binary_cols = ["contact", "housing", "loan"]
nominal_cols = ["job", "marital", "poutcome", "month", "day_of_week"]
edu_order = ["illiterate", "basic.4y", "basic.6y", "basic.9y",
             "high.school", "professional.course", "university.degree"]

# numeric part: log on campaign -> scale -> PCA (4 components)
def log_campaign(data):
    data = data.copy()
    data["campaign"] = np.log1p(data["campaign"])
    return data

num_pipe = Pipeline([
    ("log", FunctionTransformer(log_campaign)),
    ("scale", StandardScaler()),
    ("pca", PCA(n_components=4)),
])

# categorical part: fill empty values with the most common one, then encode
encode = ColumnTransformer([
    ("ord", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OrdinalEncoder(categories=[edu_order]))]), ordinal_cols),
    ("bin", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OneHotEncoder(drop="if_binary", handle_unknown="ignore"))]), binary_cols),
    ("nom", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OneHotEncoder(drop="first", handle_unknown="ignore"))]), nominal_cols),
])

# feature selection: mutual information (top 75%), then backward elimination
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
sfs = SequentialFeatureSelector(LogisticRegression(max_iter=1000, random_state=42),
                                direction="backward", scoring="roc_auc", tol=-0.0001, cv=cv, n_jobs=-1)
cat_pipe = Pipeline([
    ("encoding", encode),
    ("mi", SelectPercentile(partial(mutual_info_classif, random_state=42), percentile=75)),
    ("be", sfs),
])

# full pipeline: numeric + prev_contacted (kept as it is) + categorical
prep = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("prev", "passthrough", ["prev_contacted"]),
    ("cat", cat_pipe, ordinal_cols + binary_cols + nominal_cols),
], sparse_threshold=0)

# learn from the training set only, then apply the same steps to the test set
X_train_ready = prep.fit_transform(X_train, y_train)
X_test_ready = prep.transform(X_test)
print(X_train_ready.shape, X_test_ready.shape)

(32940, 21) (8236, 21)
